# Broadcast → .srt transcription (Kaggle T4 GPU)

Runs the `broadcast-srt` pipeline (faster-whisper `large-v3`, per-chunk language
detection) on a Darija / Arabic / French broadcast clip and verifies the output `.srt`.

**Setup:** Settings → Accelerator → **GPU T4 x2** (or P100). Internet **On** (first run
downloads the model weights). Upload this `transcription/` folder as a Kaggle **Dataset**
(it already includes `samples/jamel_debbouze_darija.mp3`).

## 1. Install

In [ ]:
!pip -q install "faster-whisper>=1.1.0"

## 2. Get the pipeline code + sample clip

Upload this whole `transcription/` repo as a **Kaggle Dataset** (Add Data → New Dataset →
upload the folder, which already contains `samples/jamel_debbouze_darija.mp3`). Then set
`ROOT` to the dataset path — Kaggle mounts it under `/kaggle/input/<dataset-name>/`.

The cell auto-discovers `ROOT` by looking for `src/transcribe.py` under `/kaggle/input`,
so usually you don't have to edit anything.

In [ ]:
import sys, glob, os

# Auto-find the uploaded repo (folder containing src/transcribe.py).
cands = glob.glob("/kaggle/input/**/src/transcribe.py", recursive=True)
ROOT = os.path.dirname(os.path.dirname(cands[0])) if cands else "/kaggle/input/transcription"
print("ROOT:", ROOT)

sys.path.insert(0, os.path.join(ROOT, "src"))
import transcribe
from srt_writer import write_srt
print("loaded:", transcribe.__file__)

## 3. Pick a clip

Defaults to the bundled Darija/French sample. Point `CLIP` at any other media file to test.

In [ ]:
CLIP = os.path.join(ROOT, "samples", "jamel_debbouze_darija.mp3")
assert glob.glob(CLIP), f"clip not found: {CLIP}"
print("clip:", CLIP)

## 4. Load the model on GPU

`large-v3` in `float16`. First run downloads the weights (~3 GB).

In [ ]:
model = transcribe.load_model("large-v3", device="cuda", compute_type="float16")

## 5. Transcribe with per-chunk language detection

`lang="auto"` detects the language per ~25 s chunk (allow-list `ar,fr,en`), so a
code-switched broadcast transcribes natively. The bundled sample mixes Darija + French.

In [ ]:
segments = transcribe.transcribe_file(
    CLIP, model,
    lang="auto", allowed=("ar", "fr", "en"),
    max_chunk_s=25.0, beam_size=5,
)
print(f"{len(segments)} cues")
for s in segments[:12]:
    print(f"[{s['start']:6.1f}-{s['end']:6.1f}] ({s['lang']}) {s['text']}")

# Per-language cue counts — sanity check that the mixed clip actually switched.
from collections import Counter
print("lang mix:", Counter(s["lang"] for s in segments))

## 6. Write the .srt

In [ ]:
out_path = write_srt(segments, "/kaggle/working/jamel_debbouze_darija.srt")
print("wrote", out_path)
print(out_path.read_text(encoding="utf-8")[:1500])

## 7. Verify the output is a valid, parseable SRT

Re-parse with the same standard-SRT contract a downstream consumer uses (e.g. the HACA
benchmark's `srt_utils.parse_srt`). If the cue count matches, the file is interoperable.
Download it from the Kaggle **Output** panel (`/kaggle/working/...srt`).

In [ ]:
import re
BLOCK = re.compile(r"(\d+)\n(\d\d:\d\d:\d\d,\d\d\d) --> (\d\d:\d\d:\d\d,\d\d\d)\n(.+?)(?=\n\n|\Z)", re.DOTALL)
text = out_path.read_text(encoding="utf-8")
cues = BLOCK.findall(text)
print(f"parsed {len(cues)} cues; first:", cues[0] if cues else None)
assert len(cues) == len(segments), "cue count mismatch — SRT format problem"